In [54]:
#README


In [55]:

#IMPORTS

import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
import pycountry




In [56]:
#INPUTS

target_ticker = "ADS.DE"

benchmark = "URTH"
start_date = "2020-12-31"
end_date = "2025-12-31"
interval = "1wk"
return_calc = "linear" #linear / log
beta_adjustment = "blume" #blume / vasicek / none
peer_group_beta_method = "median" #average / median

peer_group = ["NKE", "PUM.DE", "ONON", "DECK", "CROX"]


In [57]:
#FUNCTIONS

def download_data(tickers: list[str], start_date: str, end_date:str, interval: str):
    """Download data for the given tickers from Yahoo Finance"""
    data = yf.download(tickers = tickers, start = start_date, end = end_date, interval = interval)
    return data

def save_data(data: pd.DataFrame, benchmark: str, peer_group: list[str], start_date: str, end_date: str):
    """Save data to csv file"""
    path = f"./data/{benchmark} + {peer_group}_{start_date}_-_{end_date}.csv"
    data.to_csv(path)

def extract_col(data: pd.DataFrame, field: str):
    """Extract columns from the downloaded dataframe"""
    column_data= data[field]
    return column_data

def closest_date(ticker: str, end_date: str):
    """Find closest date relative to the valuation date in yfinance financials"""
    val_date = pd.Timestamp(end_date)
    dates = pd.to_datetime(yf.Ticker(ticker).balance_sheet.columns)
    min_diff = abs(dates - val_date).argmin()
    dates = dates.tolist()
    return dates[min_diff].strftime("%Y-%m-%d")

def get_d_e_ratio(ticker: str, end_date: str):
    """Calculates D/E ratio for given ticker from yfinance"""
    date = closest_date(ticker, end_date)
    bs = yf.Ticker(ticker).balance_sheet
    total_debt = bs.loc["Total Debt", date]
    total_equity = bs.loc["Stockholders Equity", date]
    d_e_ratio = total_debt / total_equity
    return d_e_ratio

def get_eff_tax_r(ticker: str):
    """Calculates effective tax ratio for given ticker from yfinance"""
    fin = yf.Ticker(ticker).financials
    tax_prov = fin.loc["Tax Provision"].dropna().to_list()
    pretax_inc = fin.loc["Pretax Income"].dropna().to_list()
    eff_t_rate = [a / b for a, b in zip(tax_prov, pretax_inc)]
    avg_tax = sum(eff_t_rate) / len(eff_t_rate)
    return avg_tax

def get_country_code(ticker):
    """Find the alpha_3 style country code for given ticker"""
    info = yf.Ticker(ticker).info
    country = info["country"]
    country_search = pycountry.countries.get(name=country)
    if country_search is None:
        raise ValueError(f"Country not found for {country}")
    country_code = country_search.alpha_3
    return country_code

def get_stat_tax_rate(ticker):
    """Searches the relevant statutory tax rate for given ticker from a downloaded OECD database"""
    country_code = get_country_code(ticker)
    data = pd.read_excel("./data/OECD_statutory_tax_rates/oecd_rates.xlsx", usecols="B:C", names=["country_code","tax_rate"])
    statutory_tax_rates_dict = dict(zip(data["country_code"], data["tax_rate"]))
    return statutory_tax_rates_dict[country_code]




In [58]:
#CLOSE_DATA_COLLECTION

ticker_package = peer_group + [benchmark]
data_package = download_data(tickers= ticker_package, start_date= start_date, end_date= end_date, interval= interval)
save_data(data = data_package, benchmark= benchmark, peer_group= peer_group, start_date= start_date, end_date= end_date)
close_data = extract_col(data = data_package, field= "Close")
close_data.head()


[*********************100%***********************]  5 of 6 completed


Ticker,CROX,DECK,NKE,ONON,PUM.DE,URTH
Date,,,,,,
2020-12-28,62.660000,47.796665,129.711761,NaN,85.688599,103.073906
2021-01-04,66.779999,52.404999,134.186157,NaN,82.271461,105.733055
2021-01-11,75.190002,54.333332,129.024124,NaN,81.658607,104.119225
2021-01-18,73.339996,53.313332,127.767998,NaN,79.058601,105.769730
2021-01-25,70.019997,48.663334,122.486702,NaN,75.010033,102.230309


In [59]:
#BASIC_DATA_CLEANING

close_data.columns = close_data.columns.get_level_values(0)
close_data = close_data.dropna()
close_data.head()

Ticker,CROX,DECK,NKE,ONON,PUM.DE,URTH
Date,,,,,,
2021-09-13,155.179993,72.498337,144.238403,38.950001,93.724213,120.297211
2021-09-20,156.300003,64.791664,137.940247,36.020000,92.077637,120.685394
2021-09-27,141.130005,60.911667,135.607330,30.500000,90.570618,117.829536
2021-10-04,130.399994,59.973331,140.605225,30.070000,91.631104,118.485733
2021-10-11,137.190002,59.711666,145.704559,29.719999,94.468422,121.073563


In [60]:
#LOG_RETURN_CALCULATION
if return_calc == "log":
    return_data = np.log(close_data / close_data.shift(1))
elif return_calc == "linear":
    return_data = (close_data / close_data.shift(1)) -1
else:
    raise ValueError("Return calculation must be either 'log' or 'linear'")

return_data = return_data.dropna()
return_data.head()

Ticker,CROX,DECK,NKE,ONON,PUM.DE,URTH
Date,,,,,,
2021-09-20,0.007217,-0.106301,-0.043665,-0.075225,-0.017568,0.003227
2021-09-27,-0.097057,-0.059884,-0.016913,-0.153248,-0.016367,-0.023664
2021-10-04,-0.076029,-0.015405,0.036856,-0.014098,0.011709,0.005569
2021-10-11,0.052071,-0.004363,0.036267,-0.011640,0.030965,0.021841
2021-10-18,0.091625,0.058029,0.034618,0.119112,0.014279,0.013283


In [61]:
#STATSMODELS_REGRESSION

market = return_data[benchmark]
raw_betas = []
std_errors = []

for ticker in peer_group:
    y = return_data[ticker]
    x = sm.add_constant(market)
    model = sm.OLS(y, x).fit()

    raw_betas.append(model.params[benchmark])
    std_errors.append(model.bse[benchmark])

beta_results = pd.DataFrame({
    "Ticker" : peer_group,
    "Standard Error" : std_errors,
    "Raw Betas" : raw_betas,
})

beta_results
# print(model.params["URTH"]) #raw_beta
# print(model.bse["URTH"]) #standard_error



,Ticker,Standard Error,Raw Betas
0,NKE,0.122608,1.128646
1,PUM.DE,0.173548,0.974772
2,ONON,0.198737,1.888067
3,DECK,0.148866,1.039583
4,CROX,0.200567,1.451888


In [62]:
#BETA_ADJUSTMENT

beta_mean = beta_results["Raw Betas"].mean()
beta_cross_var = beta_results["Raw Betas"].var(ddof=1)

vas_beta = []

for ticker in beta_results["Ticker"]:
    se = beta_results.loc[beta_results["Ticker"] == ticker, "Standard Error"].item()
    weight = beta_cross_var / (beta_cross_var + (se ** 2))
    r_bet = beta_results.loc[beta_results["Ticker"] == ticker, "Raw Betas"].item()
    adj_beta = weight * r_bet + (1 - weight) * beta_mean
    vas_beta.append(adj_beta)

beta_results["Blume adj. beta"] = beta_results["Raw Betas"] * (2/3) + (1 * (1/3))
beta_results["Vasicek adj. beta"] = vas_beta

beta_results


,Ticker,Standard Error,Raw Betas,Blume adj. beta,Vasicek adj. beta
0,NKE,0.122608,1.128646,1.085764,1.144627
1,PUM.DE,0.173548,0.974772,0.983181,1.030779
2,ONON,0.198737,1.888067,1.592045,1.760021
3,DECK,0.148866,1.039583,1.026388,1.074079
4,CROX,0.200567,1.451888,1.301259,1.417783


In [63]:
#D/E_RATIO

d_e_ratio = []

for ticker in beta_results["Ticker"]:
    d_e = float(get_d_e_ratio(ticker=ticker, end_date=end_date))
    d_e_ratio.append(d_e)

beta_results["D/E ratio"] = d_e_ratio

beta_results


,Ticker,Standard Error,Raw Betas,Blume adj. beta,Vasicek adj. beta,D/E ratio
0,NKE,0.122608,1.128646,1.085764,1.144627,0.742213
1,PUM.DE,0.173548,0.974772,0.983181,1.030779,1.466466
2,ONON,0.198737,1.888067,1.592045,1.760021,0.319468
3,DECK,0.148866,1.039583,1.026388,1.074079,0.150099
4,CROX,0.200567,1.451888,1.301259,1.417783,1.247870


In [64]:
#COUNTRY + STATUTORY_TAX_RATE

country_list_2 = []
stat_tax_rates_2 = []

for ticker in beta_results["Ticker"]:
    code = get_country_code(ticker)
    country_list_2.append(code)

beta_results["Country code_2"] = country_list_2

for ticker in beta_results["Ticker"]:
    stat_rate = get_stat_tax_rate(ticker)
    stat_tax_rates_2.append(stat_rate)

beta_results["Statutory tax rate"] = stat_tax_rates_2

beta_results


,Ticker,Standard Error,Raw Betas,Blume adj. beta,Vasicek adj. beta,D/E ratio,Country code_2,Statutory tax rate
0,NKE,0.122608,1.128646,1.085764,1.144627,0.742213,USA,0.255285
1,PUM.DE,0.173548,0.974772,0.983181,1.030779,1.466466,DEU,0.301330
2,ONON,0.198737,1.888067,1.592045,1.760021,0.319468,CHE,0.196078
3,DECK,0.148866,1.039583,1.026388,1.074079,0.150099,USA,0.255285
4,CROX,0.200567,1.451888,1.301259,1.417783,1.247870,USA,0.255285


In [65]:
#UNLEVERED BETA (HAMADA)

if beta_adjustment == "none":
    applied_beta = "Raw Betas"
elif beta_adjustment == "blume":
    applied_beta = "Blume adj. beta"
elif beta_adjustment == "vasicek":
    applied_beta = "Vasicek adj. beta"
else:
    raise ValueError("Beta adjustment must be either 'none' / 'blume' / 'vasicek'")

unlevered_beta_list = []

for ticker in beta_results["Ticker"]:
    levered_b = beta_results.loc[beta_results["Ticker"] == ticker, applied_beta].item()
    tax_rate = beta_results.loc[beta_results["Ticker"] == ticker, "Statutory tax rate"].item()
    peer_d_e_ratio = beta_results.loc[beta_results["Ticker"] == ticker, "D/E ratio"].item()
    unlevered_beta = levered_b /(1+(1-tax_rate)*peer_d_e_ratio)
    unlevered_beta_list.append(unlevered_beta)

beta_results["Unlevered beta"] = unlevered_beta_list

if peer_group_beta_method == "average":
    peer_group_beta = beta_results["Unlevered beta"].mean()
elif peer_group_beta_method == "median":
    peer_group_beta = beta_results["Unlevered beta"].median()
else:
    raise ValueError("Peer group beta method must be either 'average' or 'median'")




In [66]:
#Target Relevered Beta


target_country_code = get_country_code(target_ticker)
target_tax_rate = get_stat_tax_rate(target_ticker)

target_d_e = get_d_e_ratio(target_ticker, end_date)


target_relevered_beta = peer_group_beta * (1 + (1- target_tax_rate) * target_d_e)

print(target_relevered_beta)


1.1691999512036912
